# Phase 6 — Multi-Turn Memory

**What this notebook demonstrates:**

| Task | Done when |
|------|-----------|
| 5.1 Memory wired | `agent.memory.chat_memory.messages` grows with each turn |
| 5.2 Profile accumulates | After income turn, `agent.state.profile.monthly_income` is set |
| 5.3 No duplicate questions | 5-turn transcript — income asked once, never again |
| 5.4 Reset clears both | After `start over`, messages = [] and profile = all None |
| 5.5 Langfuse session view | Same `session_id` groups all turns in one trace |

Requires `OPENAI_API_KEY`. Langfuse keys are optional (traces are skipped gracefully if not set).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — Run this cell first. Safe to re-run.
# Works in Google Colab and local Jupyter equally.
# ─────────────────────────────────────────────────────────────────────────────
import sys, subprocess, os

# 1. Install required packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'langchain', 'langchain-openai', 'langchain-community', 'langchain-core',
    'langchain-chroma', 'langchain-text-splitters',
    'python-dotenv', 'chromadb', 'openai', 'tiktoken', 'langfuse'],
    check=True)
print('✓ Packages ready')

# 2. Detect runtime and set project root
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Capstone/code'
else:
    PROJECT_ROOT = os.path.abspath('..')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'✓ Project root: {PROJECT_ROOT}')

# 3. Load keys
from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

if IN_COLAB:
    from google.colab import userdata
    for key in ['OPENAI_API_KEY', 'OPENAI_BASE_URL',
                'LANGFUSE_PUBLIC_KEY', 'LANGFUSE_SECRET_KEY',
                'LANGFUSE_HOST', 'LANGFUSE_BASE_URL']:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass

if not os.environ.get('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY not found — enter it now: ')

print('✓ Environment ready')
print(f'  OPENAI_API_KEY : set')
print(f'  Langfuse keys  : {"set" if os.environ.get("LANGFUSE_PUBLIC_KEY") else "not set (optional)"}')

---
## Demo 1 — 5-Turn Session: Memory Grows, No Field Re-Asked

**Verifies tasks 5.1 + 5.3**

The customer spreads their profile across 5 turns. The agent must:
- Remember income from Turn 1 and never ask for it again
- Remember employment from Turn 2, age from Turn 3, etc.
- Only call `check_eligibility` once all fields are collected (Turn 5)

After each turn we inspect `agent.memory.chat_memory.messages` to confirm the sliding window is growing.

In [ ]:
from agent.core_agent import LoanCopilotAgent

agent = LoanCopilotAgent(phase=5)
print(f'Session ID : {agent.session_id}')
print(f'Memory window : k={agent.memory.k}')
print()

# Five turns — each adds new profile information
turns = [
    'I want a personal loan. My monthly take-home salary is Rs 55,000.',
    'I am salaried, working at a private company.',
    'I am 28 years old.',
    'I am thinking of borrowing Rs 5 lakhs.',
    'I would like to repay over 3 years. My CIBIL score is around 730.',
]

for i, user_msg in enumerate(turns, 1):
    print(f'{'='*60}')
    print(f'Turn {i} — User: {user_msg}')
    response = agent.chat(user_msg)
    print(f'Agent: {response}')
    print(f'Memory messages after turn {i}: {len(agent.memory.chat_memory.messages)}')
    print()

# Task 5.1 verification
msg_count = len(agent.memory.chat_memory.messages)
assert msg_count == 10, f'Expected 10 messages (5 user + 5 ai), got {msg_count}'
print(f'✅ Task 5.1 PASS — memory has {msg_count} messages after 5 turns (5 user + 5 AI)')

---
## Demo 2 — Profile Accumulation in SessionState

**Verifies task 5.2**

`agent.state.profile` is a `CustomerProfile` dataclass updated in-place after every turn via `_extract_profile`. Here we confirm that each field is captured correctly, so any downstream code reading `state.profile` (e.g. Phase 2 fallback, escalation tools) always has fresh data.

In [ ]:
# Use the same agent from Demo 1 — profile should already be fully populated
p = agent.state.profile

print('CustomerProfile after 5-turn session:')
print(f'  loan_product       : {p.loan_product}')
print(f'  monthly_income     : {p.monthly_income}')
print(f'  employment_type    : {p.employment_type}')
print(f'  age                : {p.age}')
print(f'  loan_amount        : {p.loan_amount}')
print(f'  tenure_months      : {p.tenure_months}')
print(f'  credit_score       : {p.credit_score}')
print()

# Task 5.2 verification — at minimum, income and product must be set
assert p.monthly_income is not None, 'monthly_income should be set after Turn 1'
assert p.loan_product is not None,   'loan_product should be set after Turn 1'
print('✅ Task 5.2 PASS — SessionState.profile accumulates across turns')
print(f'   monthly_income = {p.monthly_income:,.0f}  |  loan_product = {p.loan_product}')

---
## Demo 3 — No-Duplicate-Question Flow

**Verifies task 5.3**

We run a *new* 5-turn session where the customer gives income in Turn 1 and then provides the remaining fields.
The test checks that the agent's responses in Turns 2–5 do **not** contain any question asking for income again — because it's already in the conversation window.

In [ ]:
agent_nd = LoanCopilotAgent(phase=5)
print(f'Session ID : {agent_nd.session_id}')
print()

turns_nd = [
    'I earn Rs 50,000 per month take-home. I want a personal loan.',
    'I am 28 years old and salaried.',
    'I want Rs 5 lakhs.',
    'Repayment over 3 years please.',
    'My CIBIL score is 725.',
]

income_asked_again = False
responses = []

for i, msg in enumerate(turns_nd, 1):
    resp = agent_nd.chat(msg)
    responses.append(resp)
    print(f'Turn {i}')
    print(f'  User : {msg}')
    print(f'  Agent: {resp[:200]}')
    print()
    # Turns 2-5: agent must not re-ask for income
    if i > 1:
        income_phrases = ['income', 'salary', 'earn', 'take-home', 'take home', 'per month']
        if any(ph in resp.lower() for ph in income_phrases) and '?' in resp:
            income_asked_again = True
            print(f'  ⚠️  Possible income re-ask detected in Turn {i}')

if income_asked_again:
    print('⚠️  WARNING: income may have been re-asked (review manually)')
else:
    print('✅ Task 5.3 PASS — income question not repeated across 5 turns')

---
## Demo 4 — Reset: `start over` Clears Both Memory and Profile

**Verifies task 5.4**

After a full session, the customer types `start over`. The agent must:
1. Return a friendly reset confirmation message
2. Clear `memory.chat_memory.messages` → empty list
3. Clear all `SessionState.profile` fields → all None
4. On the very next turn, ask from scratch as if it's a brand-new session

In [ ]:
# Build up a session first
agent_reset = LoanCopilotAgent(phase=5)

agent_reset.chat('I earn 80,000 a month. I want a home loan of 60 lakhs for 20 years.')
agent_reset.chat('I am 35 years old, salaried.')

print('Before reset:')
print(f'  memory messages : {len(agent_reset.memory.chat_memory.messages)}')
print(f'  profile.monthly_income : {agent_reset.state.profile.monthly_income}')
print(f'  profile.loan_product   : {agent_reset.state.profile.loan_product}')
print()

# Customer types 'start over'
reset_response = agent_reset.chat('start over')
print(f'Agent response: {reset_response}')
print()

print('After reset:')
print(f'  memory messages : {len(agent_reset.memory.chat_memory.messages)}')
print(f'  profile.monthly_income : {agent_reset.state.profile.monthly_income}')
print(f'  profile.loan_product   : {agent_reset.state.profile.loan_product}')
print()

# Task 5.4 assertions
assert len(agent_reset.memory.chat_memory.messages) == 0, \
    'Memory should be empty after reset'
assert agent_reset.state.profile.monthly_income is None, \
    'profile.monthly_income should be None after reset'
assert agent_reset.state.profile.loan_product is None, \
    'profile.loan_product should be None after reset'
print('✅ Task 5.4 PASS — memory and profile both cleared after start over')

# Verify fresh turn after reset starts from scratch
print()
print('First turn after reset:')
fresh_response = agent_reset.chat('I want a car loan.')
print(f'  User : I want a car loan.')
print(f'  Agent: {fresh_response[:300]}')
print()
# Agent should ask for income/profile — not jump straight to eligibility
print(f'  Memory messages after first post-reset turn: {len(agent_reset.memory.chat_memory.messages)}')

---
## Demo 5 — Langfuse Session View: All Turns Under One Session ID

**Verifies task 5.5**

Every call to `_get_callbacks()` passes `session_id` as `trace_id`. Langfuse groups all traces that share the same `trace_id` into one session timeline, so the RM Dashboard (Phase 8) and evaluators can replay the full conversation in order.

This cell runs a 5-turn session, flushes the Langfuse buffer, and prints the direct link to the session trace.

In [ ]:
from deployment.config import LANGFUSE_HOST, LANGFUSE_PUBLIC_KEY
from monitoring.langfuse_logger import flush

# Fresh agent — dedicated session so it's easy to find in Langfuse
agent_lf = LoanCopilotAgent(phase=5)
session_id = agent_lf.session_id
print(f'Session ID : {session_id}')

# 5-turn conversation
session_turns = [
    'Hi, I want a home loan.',
    'My monthly income is Rs 1.2 lakh. I am salaried.',
    'I am 40 years old.',
    'I want Rs 60 lakhs for 15 years.',
    'My CIBIL score is 760.',
]

for i, msg in enumerate(session_turns, 1):
    resp = agent_lf.chat(msg)
    print(f'Turn {i}: {len(agent_lf.memory.chat_memory.messages)} messages in memory')

# Flush traces to Langfuse cloud
flush()
print()

# Print verification result
if LANGFUSE_PUBLIC_KEY:
    host = (LANGFUSE_HOST or 'https://cloud.langfuse.com').rstrip('/')
    # Langfuse v4 uses the hex session_id as trace_id
    hex_id = session_id.replace('-', '').lower()
    print(f'✅ Task 5.5 — Langfuse traces flushed for session {session_id}')
    print(f'   View session at: {host}/traces/{hex_id}')
    print(f'   All 5 turns should appear grouped under this single trace.')
else:
    print('ℹ️  Langfuse keys not configured — skipping remote trace verification.')
    print(f'   Memory has {len(agent_lf.memory.chat_memory.messages)} messages,',
          'confirming all turns were processed in one session.')

print()
print(f'Final memory snapshot ({len(agent_lf.memory.chat_memory.messages)} messages):')
for m in agent_lf.memory.chat_memory.messages:
    role = 'User ' if m.type == 'human' else 'Agent'
    print(f'  [{role}] {str(m.content)[:100]}')

---
## Demo 6 — Sliding Window: k=10 Drops Oldest Turns

**Confirms the k=10 window cap works — older turns fall out of context after 10 exchanges.**

This is a safeguard against unbounded context growth in long sessions. The agent is designed for the pre-application stage (typically 5–8 turns), so k=10 gives comfortable headroom.

In [ ]:
agent_win = LoanCopilotAgent(phase=5, memory_window=3)  # tight window for demo
print(f'Window size : k={agent_win.memory.k}')
print()

# Simulate 5 turns with a k=3 window
filler_turns = [
    'I want a home loan.',
    'My income is Rs 80,000 per month.',
    'I am salaried.',
    'I am 35 years old.',
    'I want Rs 40 lakhs for 20 years.',
]

for i, msg in enumerate(filler_turns, 1):
    agent_win.chat(msg)
    stored = len(agent_win.memory.chat_memory.messages)
    print(f'After turn {i}: {stored} messages stored (window keeps last {agent_win.memory.k * 2} messages)')

print()
# With k=3: only the last 3 human + 3 AI messages are kept = 6 max
final_count = len(agent_win.memory.chat_memory.messages)
assert final_count <= agent_win.memory.k * 2, \
    f'Window overflow: {final_count} messages > {agent_win.memory.k * 2} max'
print(f'✅ Sliding window cap confirmed — {final_count} messages retained (≤ {agent_win.memory.k * 2})')
print()
print('Note: Production agent uses k=10 (20 messages max) — more than enough for')
print('the typical 5–8 turn pre-application conversation.')

---
## Phase 6 Summary

| Task | Criterion | Status |
|------|-----------|--------|
| 5.1 Memory wired | `messages` grows with each turn | ✅ Demo 1 |
| 5.2 Profile accumulates | `monthly_income` set after Turn 1 | ✅ Demo 2 |
| 5.3 No duplicate questions | Income asked only in Turn 1 | ✅ Demo 3 |
| 5.4 Reset clears both | `messages == []` and `profile = None` after `start over` | ✅ Demo 4 |
| 5.5 Langfuse session view | All turns grouped under one `session_id` trace | ✅ Demo 5 |

**Phase 7 (Adaptive Behaviour / RLHF) is next.**